# 02e m9_pbm Nested Weight Optimisation

**Research question.** Which positive unit-sum weights for F1 bridge, F3 slope
continuity, and F4 duration give the most reliable held-out-substation result?

This notebook compares a 171-point grid and 1,000 seeded random simplex samples.
Weight and threshold selection are nested inside each outer Beta
leave-one-substation-out fold. Alpha is excluded completely. The final model
remains a deterministic physical score, not a machine-learning classifier.

**Inputs:** Beta candidate/day caches from 02b.  
**Outputs:** search results, nested outer metrics, selected weights, two figures,
the final model artifact, the Beta-B outer-fold artifact, and a manifest.  
**Expected runtime:** approximately 10-30 minutes after 02b exists.

## 1. Imports, Paths, And Search Space

Every weight is at least 0.05 and the three weights sum to one. The grid uses a
0.05 step. Random samples use seed 9 and the transformation
`0.05 + 0.85 * Dirichlet(1,1,1)`. The exact equal-weight vector is included as
a separate baseline because one third is not on the 0.05 grid.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import (  # noqa: E402
    COMPACT_FEATURE_COLUMNS,
    maximum_subset_scores,
    select_best_candidates,
)
from _m9_pbm_plotting import plot_selected_weights, plot_weight_simplex  # noqa: E402
from _m9_pbm_validation import (  # noqa: E402
    assert_heldout_absent,
    cross_validated_weight_results,
    metric_rows,
    random_simplex_weights,
    select_best_weight_result,
    select_threshold,
    simplex_grid,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "02e_m9_pbm_weight_optimisation"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)
WEIGHT_CONFIG = CONFIG["m9_pbm"]["weight_optimisation"]

grid = simplex_grid(
    step=WEIGHT_CONFIG["grid_step"],
    minimum=WEIGHT_CONFIG["minimum_weight"],
)
grid["search_origin"] = "grid"
random = random_simplex_weights(
    WEIGHT_CONFIG["random_samples"],
    minimum=WEIGHT_CONFIG["minimum_weight"],
    seed=WEIGHT_CONFIG["random_seed"],
)
random["search_origin"] = "random"
equal = pd.DataFrame(
    [{
        "weight_F1": 1 / 3,
        "weight_F3": 1 / 3,
        "weight_F4": 1 / 3,
        "search_origin": "equal",
    }]
)
WEIGHT_DEFINITIONS = pd.concat([equal, grid, random], ignore_index=True)
WEIGHT_DEFINITIONS.insert(
    0, "weight_id", [f"weight_{index:04d}" for index in range(len(WEIGHT_DEFINITIONS))]
)
WEIGHT_DEFINITIONS["score_column"] = WEIGHT_DEFINITIONS["weight_id"]
WEIGHT_MATRIX = WEIGHT_DEFINITIONS[
    ["weight_F1", "weight_F3", "weight_F4"]
].to_numpy(dtype=float)

assert len(grid) == 171 and len(random) == 1_000 and len(WEIGHT_DEFINITIONS) == 1_172
assert np.allclose(WEIGHT_MATRIX.sum(axis=1), 1.0)
assert WEIGHT_MATRIX.min() >= WEIGHT_CONFIG["minimum_weight"] - 1e-12
display(WEIGHT_DEFINITIONS.groupby("search_origin").size().rename("weight_vectors"))
display(pd.Series(WEIGHT_CONFIG, name="weight_optimisation"))

search_origin
equal        1
grid       171
random    1000
Name: weight_vectors, dtype: int64

regime            beta_only
grid_step              0.05
minimum_weight         0.05
random_samples         1000
random_seed               9
weights_sum_to          1.0
Name: weight_optimisation, dtype: object

## 2. Weighted Candidate Score

For candidate window $W$,

$$
Score_w(W)=w_1F_1(W)+w_3F_3(W)+w_4F_4(W),
$$

subject to

$$
w_1+w_3+w_4=1,\qquad w_1,w_3,w_4\geq0.05.
$$

The selected candidate is

$$
W_{d,w}^*=\operatorname*{arg\,max}_{W}Score_w(W).
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $W$ | One valid candidate correction window. |
| $d$ | One Beta substation-day. |
| $F_1(W)$ | Bridge-improvement feature. |
| $F_3(W)$ | Slope-continuity-improvement feature. |
| $F_4(W)$ | Duration-plausibility feature. |
| $w_1,w_3,w_4$ | Nonzero, unit-sum physical feature weights. |
| $Score_w(W)$ | Weighted candidate score. |
| $W_{d,w}^*$ | Candidate selected by weight vector $w$ on day $d$. |

Only Beta candidates are scored in this notebook. Alpha cannot influence the
final weights, thresholds, or Beta-B forecast-case artifact.

## 3. Build The Beta Weight-Score Cache

The 1,172 weight vectors are multiplied against F1/F3/F4 in 32-day batches.
For each Beta day and weight vector, only the maximum candidate score is kept.
The wide cache is local and reproducible; it avoids rescanning candidate rows
during the nested folds.

In [2]:
CACHE_ROOT = PATHS.intermediate / "02b_m9_pbm_candidate_features"
PARTITION_DIR = CACHE_ROOT / "_partitions"
DAY_INPUT_CACHE = CACHE_ROOT / "day_input_cache.parquet"
WEIGHT_SCORE_CACHE = OUTPUT_DIRS["intermediate"] / "beta_weight_daily_scores.parquet"
assert PARTITION_DIR.exists() and DAY_INPUT_CACHE.exists(), "Run Notebook 02b first."

if WEIGHT_SCORE_CACHE.exists() and CONFIG["execution"]["resume_validated_intermediates"]:
    daily_scores = pd.read_parquet(WEIGHT_SCORE_CACHE)
else:
    score_parts = []
    beta_candidate_paths = sorted(PARTITION_DIR.glob("beta_*_candidates.parquet"))
    assert len(beta_candidate_paths) == 8
    for candidate_path in beta_candidate_paths:
        candidates = pd.read_parquet(
            candidate_path,
            columns=[
                "dataset", "substation_id", "date", "candidate_id",
                *COMPACT_FEATURE_COLUMNS,
            ],
        )
        keys, maxima = maximum_subset_scores(
            candidates,
            feature_columns=COMPACT_FEATURE_COLUMNS,
            weight_matrix=WEIGHT_MATRIX,
            batch_days=32,
        )
        score_parts.append(
            pd.concat(
                [
                    keys,
                    pd.DataFrame(
                        maxima,
                        columns=WEIGHT_DEFINITIONS["score_column"],
                    ),
                ],
                axis=1,
            )
        )
        print(f"Scored {candidate_path.stem}", flush=True)
    daily_scores = pd.concat(score_parts, ignore_index=True)
    labels = pd.read_parquet(DAY_INPUT_CACHE)
    labels = labels.loc[labels["dataset"].eq("beta")]
    daily_scores = daily_scores.merge(
        labels[["dataset", "substation_id", "date", "true_day", "confidence"]],
        on=["dataset", "substation_id", "date"],
        how="left",
        validate="one_to_one",
    )
    write_parquet(daily_scores, WEIGHT_SCORE_CACHE)

assert len(daily_scores) == CONFIG["datasets"]["expected_substation_days"]["beta"]
assert daily_scores[WEIGHT_DEFINITIONS["score_column"]].notna().all().all()
assert daily_scores["dataset"].eq("beta").all()
display(
    pd.Series(
        {
            "beta_substation_days": len(daily_scores),
            "weight_vectors": len(WEIGHT_DEFINITIONS),
            "daily_weight_scores": len(daily_scores) * len(WEIGHT_DEFINITIONS),
        },
        name="weight_score_cache",
    )
)

Scored beta_beta_A_candidates


Scored beta_beta_B_candidates


Scored beta_beta_C_candidates


Scored beta_beta_D_candidates


Scored beta_beta_E_candidates


Scored beta_beta_F_candidates


Scored beta_beta_G_candidates


Scored beta_beta_H_candidates


beta_substation_days       2928
weight_vectors             1172
daily_weight_scores     3431616
Name: weight_score_cache, dtype: int64

## 4. Nested Beta Leave-One-Substation-Out Selection

For each outer held-out Beta substation:

1. remove all of its days and labels;
2. run inner LOSO across the other seven substations;
3. in each inner fold, select the threshold on six sure-only substations and
   evaluate the seventh;
4. aggregate inner macro-substation F1 and use precision, recall, then stable
   weight order as tie-breaks;
5. select the best equal, grid, random, and overall optimised vector;
6. retune the threshold on all seven outer-training substations; and
7. predict the outer substation exactly once.

This nesting keeps the outer substation absent from both weight and threshold
selection. Reviewer confidence filters training days only; outer Beta sure/all
metrics are created after prediction.

In [3]:
beta_substations = sorted(daily_scores["substation_id"].unique())
outer_search_parts = []
selected_rows = []
outer_decision_parts = []

for outer_heldout in beta_substations:
    inner_results = cross_validated_weight_results(
        daily_scores,
        WEIGHT_DEFINITIONS,
        excluded_substation=outer_heldout,
    )
    inner_results["selection_context"] = f"outer_holdout_{outer_heldout}"
    outer_search_parts.append(inner_results)

    strategy_candidates = {
        "equal": inner_results.loc[inner_results["search_origin"].eq("equal")],
        "grid": inner_results.loc[inner_results["search_origin"].eq("grid")],
        "random": inner_results.loc[inner_results["search_origin"].eq("random")],
        "optimised": inner_results,
    }
    for strategy, candidates in strategy_candidates.items():
        selected = select_best_weight_result(candidates)
        score_column = WEIGHT_DEFINITIONS.set_index("weight_id").loc[
            selected["weight_id"], "score_column"
        ]
        outer_training_mask = (
            daily_scores["confidence"].eq("sure")
            & ~daily_scores["substation_id"].eq(outer_heldout)
        )
        training = daily_scores.loc[
            outer_training_mask,
            ["dataset", "substation_id", "true_day"],
        ].copy()
        training["score"] = daily_scores.loc[outer_training_mask, score_column].to_numpy()
        assert_heldout_absent(training, outer_heldout)
        threshold_selection = select_threshold(training, score_column="score")

        evaluation = daily_scores.loc[daily_scores["substation_id"].eq(outer_heldout)].copy()
        evaluation["strategy"] = strategy
        evaluation["heldout_substation"] = outer_heldout
        evaluation["weight_id"] = selected["weight_id"]
        evaluation["threshold"] = threshold_selection.threshold
        evaluation["selected_score"] = evaluation[score_column]
        evaluation["predicted_day"] = evaluation["selected_score"].ge(
            threshold_selection.threshold
        )
        outer_decision_parts.append(
            evaluation[
                [
                    "dataset", "substation_id", "date", "true_day", "confidence",
                    "strategy", "heldout_substation", "weight_id", "threshold",
                    "selected_score", "predicted_day",
                ]
            ]
        )
        selected_rows.append(
            {
                "strategy": strategy,
                "heldout_substation": outer_heldout,
                "weight_id": selected["weight_id"],
                "search_origin": selected["search_origin"],
                "weight_F1": selected["weight_F1"],
                "weight_F3": selected["weight_F3"],
                "weight_F4": selected["weight_F4"],
                "inner_macro_precision": selected["inner_macro_precision"],
                "inner_macro_recall": selected["inner_macro_recall"],
                "inner_macro_f1": selected["inner_macro_f1"],
                "outer_training_substations": ",".join(
                    substation for substation in beta_substations
                    if substation != outer_heldout
                ),
                "outer_training_confidence": "sure_only",
                "outer_threshold": threshold_selection.threshold,
            }
        )
    print(f"Completed nested selection for {outer_heldout}", flush=True)

outer_search = pd.concat(outer_search_parts, ignore_index=True)
selected_weights = pd.DataFrame(selected_rows)
outer_decisions = pd.concat(outer_decision_parts, ignore_index=True)
assert len(selected_weights) == 8 * 4
assert outer_decisions.groupby("strategy").size().eq(len(daily_scores)).all()
display(
    selected_weights.loc[selected_weights["strategy"].eq("optimised")][
        [
            "heldout_substation", "search_origin", "weight_F1", "weight_F3",
            "weight_F4", "inner_macro_f1", "outer_threshold"
        ]
    ]
)

Completed nested selection for beta_A


Completed nested selection for beta_B


Completed nested selection for beta_C


Completed nested selection for beta_D


Completed nested selection for beta_E


Completed nested selection for beta_F


Completed nested selection for beta_G


Completed nested selection for beta_H


,heldout_substation,search_origin,weight_F1,weight_F3,weight_F4,inner_macro_f1,outer_threshold
3,beta_A,random,0.304733,0.119017,0.576250,0.675931,0.753130
7,beta_B,random,0.454408,0.171672,0.373921,0.685252,0.674593
11,beta_C,grid,0.250000,0.150000,0.600000,0.792863,0.770303
15,beta_D,random,0.261692,0.105020,0.633289,0.660953,0.786786
19,beta_E,grid,0.250000,0.100000,0.650000,0.662945,0.796570
23,beta_F,grid,0.250000,0.150000,0.600000,0.660667,0.770303
27,beta_G,grid,0.250000,0.150000,0.600000,0.674125,0.770303
31,beta_H,grid,0.250000,0.150000,0.600000,0.745244,0.770303


## 5. Outer-Fold Metrics And Final Full-Beta Selection

Outer predictions estimate the complete selection procedure. Separately, the
deployment artifact is fitted after evaluation by running LOSO model selection
across all eight Beta substations, choosing one weight vector, and selecting its
threshold from all sure Beta days. This full-data artifact is not used to score
the outer-fold metrics.

In [4]:
outer_metric_parts = []
for strategy, strategy_frame in outer_decisions.groupby("strategy", sort=False):
    for confidence_scope, evaluation in [
        ("beta_sure", strategy_frame.loc[strategy_frame["confidence"].eq("sure")]),
        ("beta_all", strategy_frame),
    ]:
        rows = metric_rows(evaluation)
        rows.insert(0, "confidence_scope", confidence_scope)
        rows.insert(0, "strategy", strategy)
        outer_metric_parts.append(rows)
outer_metrics = pd.concat(outer_metric_parts, ignore_index=True)

full_search = cross_validated_weight_results(daily_scores, WEIGHT_DEFINITIONS)
full_search["selection_context"] = "full_beta_model_selection"
final_selected = select_best_weight_result(full_search)
final_score_column = WEIGHT_DEFINITIONS.set_index("weight_id").loc[
    final_selected["weight_id"], "score_column"
]
final_training = daily_scores.loc[
    daily_scores["confidence"].eq("sure"),
    ["dataset", "substation_id", "true_day"],
].copy()
final_training["score"] = daily_scores.loc[
    daily_scores["confidence"].eq("sure"), final_score_column
].to_numpy()
final_threshold = select_threshold(final_training, score_column="score")

all_search = pd.concat([outer_search, full_search], ignore_index=True)
grid_results = all_search.loc[all_search["search_origin"].eq("grid")].copy()
random_results = all_search.loc[all_search["search_origin"].eq("random")].copy()
equal_results = all_search.loc[all_search["search_origin"].eq("equal")].copy()

comparison = outer_metrics.loc[
    outer_metrics["confidence_scope"].eq("beta_sure")
    & outer_metrics["aggregation"].isin(["pooled", "macro_substation"])
].copy()
stability_rows = []
optimised_weights = selected_weights.loc[selected_weights["strategy"].eq("optimised")]
for feature in ["weight_F1", "weight_F3", "weight_F4"]:
    stability_rows.append(
        {
            "weight": feature,
            "outer_fold_mean": optimised_weights[feature].mean(),
            "outer_fold_std": optimised_weights[feature].std(ddof=1),
            "outer_fold_minimum": optimised_weights[feature].min(),
            "outer_fold_maximum": optimised_weights[feature].max(),
            "full_beta_selected": final_selected[feature],
        }
    )
weight_stability = pd.DataFrame(stability_rows)

display(
    comparison[
        ["strategy", "aggregation", "support", "precision", "recall", "f1"]
    ]
)
display(
    pd.Series(
        {
            "weight_id": final_selected["weight_id"],
            "search_origin": final_selected["search_origin"],
            "weight_F1": final_selected["weight_F1"],
            "weight_F3": final_selected["weight_F3"],
            "weight_F4": final_selected["weight_F4"],
            "full_beta_loso_macro_f1": final_selected["inner_macro_f1"],
            "final_threshold": final_threshold.threshold,
        },
        name="final_full_beta_model",
    )
)

,strategy,aggregation,support,precision,recall,f1
0,equal,pooled,2310,0.832981,0.836518,0.834746
9,equal,macro_substation,2310,0.658852,0.682429,0.647038
20,grid,pooled,2310,0.891648,0.838641,0.864333
29,grid,macro_substation,2310,0.708825,0.695206,0.688060
40,random,pooled,2310,0.888631,0.813163,0.849224
49,random,macro_substation,2310,0.708181,0.682265,0.678723
60,optimised,pooled,2310,0.889145,0.817410,0.851770
69,optimised,macro_substation,2310,0.708452,0.685206,0.680646


weight_id                  weight_0069
search_origin                     grid
weight_F1                         0.25
weight_F3                         0.15
weight_F4                          0.6
full_beta_loso_macro_f1       0.693755
final_threshold               0.770303
Name: final_full_beta_model, dtype: object

## 6. Recover Outer Selected Windows For Final Evaluation

The wide score cache stores only each day's maximum score. Notebook 02g also
needs the corresponding window boundaries. Therefore each held-out Beta
partition is rescored once using only the optimised inner-selected weight for
that outer fold. The resulting 2,928-row audit contains exactly one leakage-safe
outer prediction per Beta substation-day.

In [5]:
prediction_parts = []
optimised_selection = selected_weights.loc[
    selected_weights["strategy"].eq("optimised")
].set_index("heldout_substation")
day_labels = pd.read_parquet(DAY_INPUT_CACHE)
day_labels = day_labels.loc[day_labels["dataset"].eq("beta")]
for heldout_substation in beta_substations:
    selected = optimised_selection.loc[heldout_substation]
    weights = {
        COMPACT_FEATURE_COLUMNS[0]: float(selected["weight_F1"]),
        COMPACT_FEATURE_COLUMNS[1]: float(selected["weight_F3"]),
        COMPACT_FEATURE_COLUMNS[2]: float(selected["weight_F4"]),
    }
    candidate_path = PARTITION_DIR / f"beta_{heldout_substation}_candidates.parquet"
    candidates = pd.read_parquet(candidate_path)
    best_windows = select_best_candidates(candidates, weights)
    predictions = best_windows.merge(
        day_labels.loc[
            day_labels["substation_id"].eq(heldout_substation),
            [
                "dataset", "substation_id", "date", "true_day",
                "true_interval_count", "confidence",
            ],
        ],
        on=["dataset", "substation_id", "date"],
        how="left",
        validate="one_to_one",
    )
    predictions["heldout_substation"] = heldout_substation
    predictions["weight_id"] = selected["weight_id"]
    predictions["weight_F1"] = selected["weight_F1"]
    predictions["weight_F3"] = selected["weight_F3"]
    predictions["weight_F4"] = selected["weight_F4"]
    predictions["threshold"] = selected["outer_threshold"]
    predictions["predicted_day"] = predictions["score"].ge(selected["outer_threshold"])
    predictions["confidence_margin"] = (
        predictions["score"] - selected["outer_threshold"]
    ).abs()
    prediction_parts.append(predictions)

outer_predictions = pd.concat(prediction_parts, ignore_index=True)
assert len(outer_predictions) == len(daily_scores)
assert outer_predictions.duplicated(["substation_id", "date"]).sum() == 0
assert outer_predictions["heldout_substation"].eq(
    outer_predictions["substation_id"]
).all()
PREDICTIONS_PATH = OUTPUT_DIRS["intermediate"] / "nested_outer_predictions.parquet"
write_parquet(outer_predictions, PREDICTIONS_PATH)
display(outer_predictions.head())

,dataset,substation_id,date,candidate_id,left_slot,right_slot,duration_slots,duration_hours,solar_peak_slot,F1_bridge_improvement,...,true_interval_count,confidence,heldout_substation,weight_id,weight_F1,weight_F3,weight_F4,threshold,predicted_day,confidence_margin
0,beta,beta_A,2023-10-01,265,39,65,27,6.75,53,-0.392425,...,0,sure,beta_A,weight_0350,0.304733,0.119017,0.57625,0.75313,False,0.353234
1,beta,beta_A,2023-10-02,534,46,53,8,2.00,50,0.450243,...,0,sure,beta_A,weight_0350,0.304733,0.119017,0.57625,0.75313,False,0.034731
2,beta,beta_A,2023-10-03,525,46,69,24,6.00,51,-0.875537,...,0,sure,beta_A,weight_0350,0.304733,0.119017,0.57625,0.75313,False,0.397087
3,beta,beta_A,2023-10-04,559,46,55,10,2.50,49,-0.319122,...,0,sure,beta_A,weight_0350,0.304733,0.119017,0.57625,0.75313,False,0.283820
4,beta,beta_A,2023-10-05,321,40,62,23,5.75,52,0.817475,...,22,sure,beta_A,weight_0350,0.304733,0.119017,0.57625,0.75313,True,0.175785


## 7. Write Tables, Figures, And Model Artifacts

The Beta-B artifact is the exact outer-fold model later applied to Gamma. Its
training-substation list excludes Beta B, and its metadata states that neither
Alpha nor Beta-B labels were used. The separate full-Beta model artifact is the
post-evaluation model intended for general future inference.

In [6]:
GRID_PATH = OUTPUT_DIRS["metrics"] / "01_grid_search_results.csv"
RANDOM_PATH = OUTPUT_DIRS["metrics"] / "02_random_search_results.csv"
OUTER_PATH = OUTPUT_DIRS["metrics"] / "03_nested_outer_fold_metrics.csv"
SELECTED_PATH = OUTPUT_DIRS["metrics"] / "04_selected_weights_and_thresholds.csv"
COMPARISON_PATH = OUTPUT_DIRS["tables"] / "table01_equal_vs_grid_vs_random.csv"
STABILITY_PATH = OUTPUT_DIRS["tables"] / "table02_weight_stability.csv"
write_csv(grid_results, GRID_PATH)
write_csv(random_results, RANDOM_PATH)
write_csv(outer_metrics, OUTER_PATH)
write_csv(selected_weights, SELECTED_PATH)
write_csv(comparison, COMPARISON_PATH)
write_csv(weight_stability, STABILITY_PATH)

FIGURE_SIMPLEX = OUTPUT_DIRS["figures"] / "fig01_weight_simplex_performance.png"
FIGURE_FOLDS = OUTPUT_DIRS["figures"] / "fig02_selected_weights_by_fold.png"
plot_weight_simplex(full_search, FIGURE_SIMPLEX)
plot_selected_weights(selected_weights, FIGURE_FOLDS)

final_model_payload = {
    "status": "publication_ready",
    "model": "m9_pbm_compact_optimised_physical_score",
    "model_family": "deterministic_non_ml",
    "features": COMPACT_FEATURE_COLUMNS,
    "weights": {
        "F1_bridge_improvement": float(final_selected["weight_F1"]),
        "F3_slope_continuity_improvement": float(final_selected["weight_F3"]),
        "F4_duration_plausibility": float(final_selected["weight_F4"]),
    },
    "threshold": float(final_threshold.threshold),
    "selection_origin": str(final_selected["search_origin"]),
    "selection_objective": "Beta sure macro-substation F1",
    "training_substations": beta_substations,
    "training_confidence": "sure_only",
    "alpha_used": False,
    "random_seed": WEIGHT_CONFIG["random_seed"],
    "candidate_windows": CONFIG["m9_pbm"]["candidate_windows"],
}
FINAL_MODEL_PATH = write_manifest(
    PATHS, "02e_m9_pbm_final_model.json", final_model_payload
)

beta_b_selected = optimised_selection.loc["beta_B"]
beta_b_training = beta_b_selected["outer_training_substations"].split(",")
assert "beta_B" not in beta_b_training
beta_b_payload = {
    "status": "publication_ready",
    "model": "m9_pbm_beta_B_outer_fold",
    "model_family": "deterministic_non_ml",
    "heldout_substation": "beta_B",
    "features": COMPACT_FEATURE_COLUMNS,
    "weights": {
        "F1_bridge_improvement": float(beta_b_selected["weight_F1"]),
        "F3_slope_continuity_improvement": float(beta_b_selected["weight_F3"]),
        "F4_duration_plausibility": float(beta_b_selected["weight_F4"]),
    },
    "threshold": float(beta_b_selected["outer_threshold"]),
    "training_substations": beta_b_training,
    "training_confidence": "sure_only",
    "heldout_labels_used": False,
    "alpha_used": False,
    "candidate_windows": CONFIG["m9_pbm"]["candidate_windows"],
}
BETA_B_MODEL_PATH = write_manifest(
    PATHS, "02e_m9_pbm_beta_B_outer_fold_model.json", beta_b_payload
)
display(FIGURE_SIMPLEX)
display(FIGURE_FOLDS)
display(pd.Series(beta_b_payload, name="Beta-B outer-fold model"))

WindowsPath('C:/Users/z5404477/Documents/PyNRPF/publication/2_journal_article/outputs/figures/02e_m9_pbm_weight_optimisation/fig01_weight_simplex_performance.png')

WindowsPath('C:/Users/z5404477/Documents/PyNRPF/publication/2_journal_article/outputs/figures/02e_m9_pbm_weight_optimisation/fig02_selected_weights_by_fold.png')

status                                                  publication_ready
model                                            m9_pbm_beta_B_outer_fold
model_family                                         deterministic_non_ml
heldout_substation                                                 beta_B
features                [F1_bridge_improvement, F3_slope_continuity_im...
weights                 {'F1_bridge_improvement': 0.4544077203861563, ...
threshold                                                        0.674593
training_substations    [beta_A, beta_C, beta_D, beta_E, beta_F, beta_...
training_confidence                                             sure_only
heldout_labels_used                                                 False
alpha_used                                                          False
candidate_windows       {'slots_per_day': 96, 'slot_minutes': 15, 'sca...
Name: Beta-B outer-fold model, dtype: object

## 8. Interpretation, Leakage Statement, And Manifest

The outer-fold optimised result is the performance estimate for the full nested
selection procedure. The full-Beta artifact is fitted only after that estimate
is complete. Beta labels informed feature-family development, so these results
remain development evidence rather than independent external validation.

No Alpha data enters this notebook. The Gamma case-study model excludes Beta B
from inner weight selection, final threshold selection, and every training-label
operation.

In [7]:
MANIFEST_OUTPUTS = [
    GRID_PATH, RANDOM_PATH, OUTER_PATH, SELECTED_PATH, COMPARISON_PATH,
    STABILITY_PATH, FIGURE_SIMPLEX, FIGURE_FOLDS, FINAL_MODEL_PATH,
    BETA_B_MODEL_PATH,
]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[
        PATHS.config,
        CACHE_ROOT / "candidate_feature_cache.parquet",
        DAY_INPUT_CACHE,
    ],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "weight_vectors": len(WEIGHT_DEFINITIONS),
        "outer_selected_models": len(selected_weights),
        "outer_predictions": len(outer_predictions),
        "grid_result_rows": len(grid_results),
        "random_result_rows": len(random_results),
    },
)
manifest["local_intermediates"] = [
    str(WEIGHT_SCORE_CACHE.relative_to(PATHS.article)),
    str(PREDICTIONS_PATH.relative_to(PATHS.article)),
]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)
inventory = pd.DataFrame(
    {"path": [*MANIFEST_OUTPUTS, WEIGHT_SCORE_CACHE, PREDICTIONS_PATH, MANIFEST_PATH]}
)
inventory["exists"] = inventory["path"].map(Path.exists)
inventory["bytes"] = inventory["path"].map(lambda path: path.stat().st_size)
display(inventory)
assert inventory["exists"].all() and inventory["bytes"].gt(0).all()

,path,exists,bytes
0,C:\Users\z5404477\Documents\PyNRPF\publication...,True,312407
1,C:\Users\z5404477\Documents\PyNRPF\publication...,True,2113115
2,C:\Users\z5404477\Documents\PyNRPF\publication...,True,8266
3,C:\Users\z5404477\Documents\PyNRPF\publication...,True,7200
4,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1002
5,C:\Users\z5404477\Documents\PyNRPF\publication...,True,361
6,C:\Users\z5404477\Documents\PyNRPF\publication...,True,523651
7,C:\Users\z5404477\Documents\PyNRPF\publication...,True,88820
8,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1121
9,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1062
